In [20]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score,r2_score,classification_report,confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import pandas as pd
import matplotlib.pyplot as plt

In [21]:
data=load_breast_cancer()
print(data.keys())

df=pd.DataFrame(data.data,columns=data.feature_names)
df["target"]=data.target
df.to_csv("breast_cancer_detect.csv",index=False)



dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])


In [22]:
breast_cancer=pd.read_csv("breast_cancer_detect.csv")
print("FIRST 5 DATAS\n",breast_cancer.head())
print("INFORMATION")
breast_cancer.info()
print("DESCRIBE\n",breast_cancer.describe())

FIRST 5 DATAS
    mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \

In [23]:
breast_cancer.isna().sum()

mean radius                0
mean texture               0
mean perimeter             0
mean area                  0
mean smoothness            0
mean compactness           0
mean concavity             0
mean concave points        0
mean symmetry              0
mean fractal dimension     0
radius error               0
texture error              0
perimeter error            0
area error                 0
smoothness error           0
compactness error          0
concavity error            0
concave points error       0
symmetry error             0
fractal dimension error    0
worst radius               0
worst texture              0
worst perimeter            0
worst area                 0
worst smoothness           0
worst compactness          0
worst concavity            0
worst concave points       0
worst symmetry             0
worst fractal dimension    0
target                     0
dtype: int64

In [24]:
breast_cancer.duplicated().sum()

np.int64(0)

In [25]:
breast_cancer.columns

Index(['mean radius', 'mean texture', 'mean perimeter', 'mean area',
       'mean smoothness', 'mean compactness', 'mean concavity',
       'mean concave points', 'mean symmetry', 'mean fractal dimension',
       'radius error', 'texture error', 'perimeter error', 'area error',
       'smoothness error', 'compactness error', 'concavity error',
       'concave points error', 'symmetry error', 'fractal dimension error',
       'worst radius', 'worst texture', 'worst perimeter', 'worst area',
       'worst smoothness', 'worst compactness', 'worst concavity',
       'worst concave points', 'worst symmetry', 'worst fractal dimension',
       'target'],
      dtype='object')

In [26]:
x=breast_cancer.drop(["target"],axis=1)
y=breast_cancer["target"]

In [27]:
preprocess=ColumnTransformer([
    ("scaler",StandardScaler(),x.columns),
])

In [28]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [29]:
pipe_lr=Pipeline([
    ["preprocess",preprocess],
    ["logistic",LogisticRegression()]
])
pipe_knn=Pipeline([
    ["preprocess",preprocess],
    ["knn",KNeighborsClassifier(n_neighbors=3)]
])
pipe_dt=Pipeline([
    ["preprocess",preprocess],
    ["decision",DecisionTreeClassifier(max_depth=5,
                                       max_leaf_nodes=3,
                                       min_samples_leaf=5,
                                       min_samples_split=5)]
])

pipe_rf=Pipeline([
    ["preprocess",preprocess],
    ["random_forest",RandomForestClassifier(max_depth=5,
                                       max_leaf_nodes=3,
                                       min_samples_leaf=5,
                                       min_samples_split=5,
                                       ccp_alpha=0.02)]
])

In [30]:
models={
    "logistic":pipe_lr,
    "knn":pipe_knn,
    "decision_tree":pipe_dt,
    "random":pipe_rf
}

In [31]:
for name,model in models.items():
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    print(name,"training data: ",model.score(x_train,y_train))
    print(name,"testing data: ",model.score(x_test,y_test))
    acc=accuracy_score(y_test,y_pred)
    print(f"{name} Accuracy: {acc:.4f}")

logistic training data:  0.9868131868131869
logistic testing data:  0.9736842105263158
logistic Accuracy: 0.9737
knn training data:  0.9846153846153847
knn testing data:  0.9473684210526315
knn Accuracy: 0.9474
decision_tree training data:  0.9230769230769231
decision_tree testing data:  0.9122807017543859
decision_tree Accuracy: 0.9123
random training data:  0.9582417582417583
random testing data:  0.9649122807017544
random Accuracy: 0.9649
